In [1]:

import tensorflow as tf
import numpy as np

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 1. Dataset directory
DATA_DIR = "data/flower_photos"

# 2. Load all images from the directory
dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

class_names = dataset.class_names
print("Classes:", class_names)

# 3. Load pretrained MobileNetV2
model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    pooling="avg"
)

# 4. Extract embeddings for all images
all_embeddings = []
all_labels = []

for images, labels in dataset:
    # MobileNetV2 expects pixels scaled to [-1, 1]
    images = preprocess_input(images)

    # Generate embeddings
    embeddings = model.predict(images, verbose=0)

    all_embeddings.append(embeddings)
    all_labels.append(labels.numpy())

# 5. Combine all batches
X = np.concatenate(all_embeddings, axis=0)
y = np.concatenate(all_labels, axis=0)

print("Embedding shape:", X.shape)
print("Labels shape:", y.shape)

# 6. Save embeddings and labels
np.save("flower_embeddings.npy", X)
np.save("flower_labels.npy", y)
np.save("flower_class_names.npy", np.array(class_names))

print("Embeddings saved successfully!")


Found 3670 files belonging to 5 classes.
Classes: ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


C:\Users\windows\AppData\Local\Temp\ipykernel_14064\715554008.py:22: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Embedding shape: (3670, 1280)
Labels shape: (3670,)
Embeddings saved successfully!


In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Load your flower dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/flower_photos",
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset="training",
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/flower_photos",
    image_size=(224, 224),
    batch_size=32,
    validation_split=0.2,
    subset="validation",
    seed=42
)

# Load pretrained MobileNetV2
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3),
    pooling="avg"
)

# Freeze MobileNetV2 initially
base_model.trainable = False

# Add our own classifier
inputs = layers.Input(shape=(224, 224, 3))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the classifier using your flower images
model.fit(train_ds, validation_data=val_ds, epochs=10)

Found 3670 files belonging to 5 classes.
Using 2936 files for training.
Found 3670 files belonging to 5 classes.
Using 734 files for validation.
Epoch 1/10


c:\Users\windows\Desktop\projects\computer-vision\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


92/92 ━━━━━━━━━━━━━━━━━━━━ 29s 278ms/step - accuracy: 0.7050 - loss: 0.7831 - val_accuracy: 0.8474 - val_loss: 0.4763
Epoch 2/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 262ms/step - accuracy: 0.8512 - loss: 0.4155 - val_accuracy: 0.8706 - val_loss: 0.3968
Epoch 3/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 264ms/step - accuracy: 0.8944 - loss: 0.3254 - val_accuracy: 0.8597 - val_loss: 0.4077
Epoch 4/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 25s 267ms/step - accuracy: 0.9070 - loss: 0.2743 - val_accuracy: 0.8747 - val_loss: 0.3703
Epoch 5/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 261ms/step - accuracy: 0.9200 - loss: 0.2450 - val_accuracy: 0.8910 - val_loss: 0.3481
Epoch 6/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 261ms/step - accuracy: 0.9271 - loss: 0.2215 - val_accuracy: 0.8978 - val_loss: 0.3411
Epoch 7/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 262ms/step - accuracy: 0.9366 - loss: 0.1984 - val_accuracy: 0.8924 - val_loss: 0.3365
Epoch 8/10
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 263ms/step - accuracy: 0.9373 - loss: 0.1878 - val_accuracy: 0.901